# Sistema de Diagnóstico Automático com NLP

## Objetivo
Utilizar técnicas de Processamento de Linguagem Natural para:
1. Extrair sintomas de frases em linguagem natural
2. Normalizar os sintomas através de lematização
3. Mapear os sintomas para diagnósticos usando base de conhecimento
4. Sugerir diagnóstico mais provável com score de confiança

## Técnicas a serem aplicadas
- **Tokenização**: Dividir texto em palavras
- **Lematização**: Reduzir palavras à forma canônica
- **NER (Named Entity Recognition)**: Reconhecimento de entidades
- **Análise Sintática**: Extrair relacionamentos entre termos
- **Similarity Matching**: Encontrar sintomas similares no mapa

In [116]:
# Instalação das bibliotecas necessárias
import subprocess
import sys

# Instalar NLTK e SPACY
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nltk", "spacy", "pandas"])

# Baixar modelo português do SPACY (sem a flag -q)
subprocess.check_call([sys.executable, "-m", "spacy", "download", "pt_core_news_sm"])

print("✓ Bibliotecas instaladas com sucesso!")

0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 10.5 MB/s  0:00:01 eta 0:00:010:0101
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
✓ Bibliotecas instaladas com sucesso!


## 1. Imports e Configuração

In [117]:
import pandas as pd
import nltk
import spacy
import re
from collections import Counter
from pathlib import Path
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')

# Download de recursos NLTK necessários
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# Carregando modelo português do SPACY
nlp = spacy.load('pt_core_news_sm')

# Configurar stemmer português
stemmer = SnowballStemmer('portuguese')
stop_words = set(stopwords.words('portuguese'))

print("✓ Bibliotecas carregadas com sucesso!")

✓ Bibliotecas carregadas com sucesso!


In [118]:
# Configuração: Carregar paths do arquivo de configuração centralizado
import sys
from pathlib import Path

# Em Jupyter Notebook, o cwd pode não estar no diretório do projeto
# Assumir que o projeto está em um diretório conhecido
current_dir = Path.cwd()

# Tentar encontrar o diretório scripts do projeto
# O projeto é: chap01-phase02-automate-diagnostics/
# Se cwd é /Users/amanda/Documents/FIAP/projetos
# Então scripts está em: chap01-phase02-automate-diagnostics/scripts

scripts_dir = current_dir / 'chap01-phase02-automate-diagnostics' / 'scripts'
if not scripts_dir.exists():
    # Se já estamos dentro do projeto
    scripts_dir = current_dir / 'scripts'
    if not scripts_dir.exists():
        # Tente ir um nível acima
        scripts_dir = current_dir.parent / 'scripts'

print(f"Procurando scripts em: {scripts_dir}")
print(f"Scripts encontrado? {scripts_dir.exists()}\n")

if not scripts_dir.exists():
    raise FileNotFoundError(f"Diretório de scripts não encontrado em: {scripts_dir}")

sys.path.insert(0, str(scripts_dir))

# Importar configuração
import config
from config import DOCUMENTS_DIR, DIAGNOSTICS_CSV, SYMPTOMS_TXT, RESULTS_CSV

# Validar configuração
try:
    config.validate_config()
    print("Configuração carregada com sucesso!")
except FileNotFoundError as e:
    print(f"Erro na configuração: {e}")
    raise

Procurando scripts em: /Users/amanda/Documents/FIAP/projetos/chap01-phase02-automate-diagnostics/scripts
Scripts encontrado? True

✅ Configuração validada com sucesso!
Configuração carregada com sucesso!


## 2. Carregamento de Dados

In [119]:
# Definir caminhos usando configuração centralizada
# Usar paths já definidos na configuração importada

# Ler arquivo de sintomas
with open(SYMPTOMS_TXT, 'r', encoding='utf-8') as f:
    sentences = [line.strip() for line in f if line.strip()]

print(f"✓ Carregadas {len(sentences)} frases de sintomas\n")

# Exibir primeiras frases
print("Primeiras 3 frases:\n")
for i, sent in enumerate(sentences[:3], 1):
    print(f"{i}. {sent}\n")

✓ Carregadas 30 frases de sintomas

Primeiras 3 frases:

1. Sinto uma dor forte no peito há algumas horas, com sensação de aperto e suor excessivo, o que me deixou sem conseguir realizar minhas atividades.

2. Estou com dor no tórax em forma de pressão desde hoje cedo, junto com falta de ar e cansaço extremo ao mínimo esforço.

3. Tenho uma dor intensa no peito que irradia para o braço esquerdo há algumas horas, além de náuseas e suor frio.



In [120]:
# Carregar mapa de diagnósticos usando configuração centralizada
df_diagnostics = pd.read_csv(DIAGNOSTICS_CSV)

print("Mapa de Conhecimento (primeiras 10 linhas):\n")
print(df_diagnostics.head(10))

print(f"\n✓ Total de relações: {len(df_diagnostics)}")
print(f"\nDiagnósticos disponíveis: {df_diagnostics['Doenca Associada'].unique()}")

Mapa de Conhecimento (primeiras 10 linhas):

            Sintoma 1              Sintoma 2 Doenca Associada
0        dor no peito        aperto no peito          Infarto
1        dor no tórax       pressão no tórax          Infarto
2        dor no peito         suor excessivo          Infarto
3        dor no peito  irradiação para braço          Infarto
4        dor no peito                náuseas          Infarto
5     aperto no peito            falta de ar          Infarto
6   tosse persistente             febre alta        Pneumonia
7  tosse com secreção                  febre        Pneumonia
8         falta de ar        tosse constante        Pneumonia
9          febre alta              calafrios        Pneumonia

✓ Total de relações: 31

Diagnósticos disponíveis: <StringArray>
[                'Infarto',               'Pneumonia',
               'Enxaqueca',                'Gastrite',
  'Insuficiência Cardíaca', 'Transtorno de Ansiedade']
Length: 6, dtype: str


## 3. Funções de Processamento de Texto

In [121]:
def lemmatize_text(text):
    """
    Lematiza um texto usando SPACY.
    Reduz as palavras à sua forma canônica.
    
    Ex: "comendo", "comeu", "comida" -> "comer" (lemma)
    """
    doc = nlp(text.lower())
    lemmas = [token.lemma_ for token in doc if not token.is_punct]
    return lemmas

def remove_stopwords(tokens):
    """
    Remove palavras muito comuns que não agregam significado.
    Ex: "o", "a", "que", "de", etc.
    """
    return [token for token in tokens if token not in stop_words and len(token) > 2]

def extract_symptoms_from_sentence(sentence):
    """
    Extrai possíveis sintomas de uma frase usando análise sintática.
    Relaciona substantivos e adjetivos com termos de dor/sensação.
    """
    doc = nlp(sentence.lower())
    
    symptoms = []
    
    # Buscar termos relacionados a sintomas (multi-palavra)
    text_lower = sentence.lower()
    
    # Padrões comuns de sintomas
    symptom_patterns = [
        r'dor\s+(?:de\s+)?(?:\w+\s+)*(?:no|na|na|nos|nas)\s+\w+',
        r'(?:sensação|sensibilidade|falta|excesso|inchaço|queimação|azia)\s+(?:\w+\s+)*(?:de|no|na)',
        r'(?:tosse|febre|náuseas?|tontura|fraqueza|fadiga|cansaço|palpitações?|ansiedade)',
        r'(?:pressão|aperto|dificuldade)\s+(?:\w+\s+)*(?:no|na|para)',
    ]
    
    for pattern in symptom_patterns:
        matches = re.findall(pattern, text_lower, re.IGNORECASE)
        symptoms.extend(matches)
    
    # Lematizar sintomas extraídos
    lemmatized_symptoms = []
    for symptom in symptoms:
        lemmas = lemmatize_text(symptom)
        lemmatized_symptoms.extend(lemmas)
    
    # Remover duplicatas mantendo ordem
    seen = set()
    unique_symptoms = []
    for symp in lemmatized_symptoms:
        if symp not in seen and symp not in stop_words:
            unique_symptoms.append(symp)
            seen.add(symp)
    
    return unique_symptoms

print("✓ Funções de processamento definidas")

✓ Funções de processamento definidas


## 4. Teste de Extração de Sintomas

In [122]:
# Testar extração em algumas frases
print("Teste de extração de sintomas:\n")

for i, sentence in enumerate(sentences[:3], 1):
    symptoms = extract_symptoms_from_sentence(sentence)
    print(f"Frase {i}: {sentence[:60]}...")
    print(f"Sintomas extraídos: {symptoms}")
    print()

Teste de extração de sintomas:

Frase 1: Sinto uma dor forte no peito há algumas horas, com sensação ...
Sintomas extraídos: ['dor', 'forte', 'em o', 'peito', 'sensação']

Frase 2: Estou com dor no tórax em forma de pressão desde hoje cedo, ...
Sintomas extraídos: ['dor', 'em o', 'tórax', 'falta', 'cansaço']

Frase 3: Tenho uma dor intensa no peito que irradia para o braço esqu...
Sintomas extraídos: ['dor', 'intenso', 'em o', 'peito', 'náuseo']



## 5. Sistema de Matching de Sintomas

In [123]:
def similarity(a, b):
    """
    Calcula similaridade entre duas strings usando SequenceMatcher.
    Útil para encontrar sintomas similares mesmo com pequenas variações.
    """
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def find_matching_symptom(extracted_symptom, symptom_database, threshold=0.6):
    """
    Encontra sintoma mais similar na base de dados.
    Usa similaridade fuzzy para lidar com variações de escrita.
    """
    best_match = None
    best_score = threshold
    
    for db_symptom in symptom_database:
        score = similarity(extracted_symptom, db_symptom)
        if score > best_score:
            best_score = score
            best_match = db_symptom
    
    return best_match, best_score

# Construir banco de dados de sintomas do CSV
all_symptoms = list(set(
    list(df_diagnostics['Sintoma 1'].unique()) + 
    list(df_diagnostics['Sintoma 2'].unique())
))

print(f"Base de sintomas criada com {len(all_symptoms)} sintomas únicos\n")
print("Amostra de sintomas conhecidos:")
print(all_symptoms[:10])

Base de sintomas criada com 44 sintomas únicos

Amostra de sintomas conhecidos:
['suor excessivo', 'tosse com secreção', 'dor no estômago', 'pressão no tórax', 'alimentação', 'estufamento', 'pensamentos acelerados', 'calafrios', 'dor de cabeça', 'desconforto abdominal']


## 6. Sistema de Diagnóstico

In [124]:
def suggest_diagnosis(extracted_symptoms, diagnostics_df, symptom_database):
    """
    Sugere diagnóstico baseado nos sintomas extraídos.
    
    Processo:
    1. Para cada sintoma extraído, encontra matches na base
    2. Busca diagnósticos associados aos sintomas matchados
    3. Calcula score de confiança baseado em quantidade de matches
    4. Retorna diagnóstico com maior score
    """
    
    diagnosis_scores = {}
    matched_symptoms_detail = []
    
    # Para cada sintoma extraído
    for extracted_symp in extracted_symptoms:
        # Encontrar match na base
        matched_symp, score = find_matching_symptom(extracted_symp, symptom_database, threshold=0.5)
        
        if matched_symp and score > 0.5:
            matched_symptoms_detail.append({
                'extracted': extracted_symp,
                'matched': matched_symp,
                'similarity_score': score
            })
            
            # Buscar diagnósticos associados a este sintoma
            relevant_rows = diagnostics_df[
                (diagnostics_df['Sintoma 1'].str.lower() == matched_symp.lower()) |
                (diagnostics_df['Sintoma 2'].str.lower() == matched_symp.lower())
            ]
            
            # Atualizar scores de diagnóstico
            for diagnosis in relevant_rows['Doenca Associada'].unique():
                if diagnosis not in diagnosis_scores:
                    diagnosis_scores[diagnosis] = 0
                diagnosis_scores[diagnosis] += score
    
    # Calcular confiança normalizada
    if diagnosis_scores:
        max_score = max(diagnosis_scores.values())
        normalized_scores = {
            diag: min((score / max_score) * 100, 100) 
            for diag, score in diagnosis_scores.items()
        }
        
        # Ordenar por score
        ranked_diagnosis = sorted(normalized_scores.items(), key=lambda x: x[1], reverse=True)
        
        return {
            'top_diagnosis': ranked_diagnosis[0][0],
            'confidence': ranked_diagnosis[0][1],
            'all_candidates': ranked_diagnosis,
            'matched_symptoms': matched_symptoms_detail,
            'num_symptoms_found': len(matched_symptoms_detail)
        }
    else:
        return {
            'top_diagnosis': 'Inconclusivo',
            'confidence': 0,
            'all_candidates': [],
            'matched_symptoms': [],
            'num_symptoms_found': 0
        }

print("✓ Sistema de diagnóstico implementado")

✓ Sistema de diagnóstico implementado


## 7. Processamento Completo de Todas as Frases

In [125]:
# Processar todas as frases e gerar diagnósticos
results = []

print("Processando frases...\n")

for idx, sentence in enumerate(sentences, 1):
    # Extrair sintomas
    extracted_symptoms = extract_symptoms_from_sentence(sentence)
    
    # Sugerir diagnóstico
    diagnosis_result = suggest_diagnosis(extracted_symptoms, df_diagnostics, all_symptoms)
    
    results.append({
        'frase_id': idx,
        'frase': sentence,
        'sintomas_extraidos': extracted_symptoms,
        'num_sintomas': len(extracted_symptoms),
        'diagnostico_sugerido': diagnosis_result['top_diagnosis'],
        'confianca': diagnosis_result['confidence'],
        'candidates': diagnosis_result['all_candidates'],
        'match_details': diagnosis_result['matched_symptoms']
    })
    
    # Mostrar progresso
    if idx % 5 == 0:
        print(f"✓ Processadas {idx}/{len(sentences)} frases")

print(f"\n✓ Processamento concluído!")

Processando frases...

✓ Processadas 5/30 frases
✓ Processadas 10/30 frases
✓ Processadas 15/30 frases
✓ Processadas 20/30 frases
✓ Processadas 25/30 frases
✓ Processadas 30/30 frases

✓ Processamento concluído!


## 8. Análise de Resultados

In [126]:
# Criar DataFrame com resultados
df_results = pd.DataFrame([
    {
        'ID': r['frase_id'],
        'Frase': r['frase'][:60] + '...' if len(r['frase']) > 60 else r['frase'],
        'Num Sintomas': r['num_sintomas'],
        'Diagnóstico': r['diagnostico_sugerido'],
        'Confiança (%)': f"{r['confianca']:.1f}%"
    }
    for r in results
])

print("\n" + "="*100)
print("RESULTADOS DO DIAGNÓSTICO AUTOMÁTICO")
print("="*100)
print(df_results.to_string(index=False))
print("="*100)


RESULTADOS DO DIAGNÓSTICO AUTOMÁTICO
 ID                                                           Frase  Num Sintomas             Diagnóstico Confiança (%)
  1 Sinto uma dor forte no peito há algumas horas, com sensação ...             5  Insuficiência Cardíaca        100.0%
  2 Estou com dor no tórax em forma de pressão desde hoje cedo, ...             5  Insuficiência Cardíaca        100.0%
  3 Tenho uma dor intensa no peito que irradia para o braço esqu...             5                 Infarto        100.0%
  4 Sinto um aperto no peito constante desde ontem, acompanhado ...             4               Pneumonia        100.0%
  5 Estou com dor no peito em pontada há algumas horas, junto co...             5 Transtorno de Ansiedade        100.0%
  6 Estou com tosse persistente há cinco dias, febre alta e difi...             3               Pneumonia        100.0%
  7 Tenho febre, tosse com secreção e dor no peito ao respirar h...             5               Pneumonia        100.0%
  

## 9. Análise Detalhada por Diagnóstico

In [127]:
# Agrupar por diagnóstico
diagnosis_groups = {}
for r in results:
    diag = r['diagnostico_sugerido']
    if diag not in diagnosis_groups:
        diagnosis_groups[diag] = []
    diagnosis_groups[diag].append(r)

print("\n" + "="*100)
print("DIAGNÓSTICOS IDENTIFICADOS E FRASES ASSOCIADAS")
print("="*100)

for diagnosis, cases in sorted(diagnosis_groups.items()):
    avg_confidence = sum([c['confianca'] for c in cases]) / len(cases)
    print(f"\n📋 {diagnosis.upper()}")
    print(f"   Casos: {len(cases)} | Confiança média: {avg_confidence:.1f}%")
    print(f"   {'-'*90}")
    
    for case in cases:
        print(f"   Frase {case['frase_id']:2d}: (Conf: {case['confianca']:5.1f}%) {case['frase'][:70]}...")
        if case['match_details']:
            print(f"              Sintomas: {', '.join([m['matched'] for m in case['match_details']])}")


DIAGNÓSTICOS IDENTIFICADOS E FRASES ASSOCIADAS

📋 ENXAQUECA
   Casos: 2 | Confiança média: 100.0%
   ------------------------------------------------------------------------------------------
   Frase 12: (Conf: 100.0%) Estou com uma dor pulsante na cabeça desde ontem, acompanhada de enjoo...
              Sintomas: dor pulsante na cabeça, dor de cabeça
   Frase 14: (Conf: 100.0%) Sinto dor na cabeça em um dos lados há dois dias, junto com náuseas e ...
              Sintomas: dor de cabeça, náuseas

📋 GASTRITE
   Casos: 2 | Confiança média: 100.0%
   ------------------------------------------------------------------------------------------
   Frase 17: (Conf: 100.0%) Estou com queimação no estômago e desconforto abdominal desde ontem, p...
              Sintomas: queimação, dor no estômago, desconforto abdominal, dor abdominal
   Frase 20: (Conf: 100.0%) Estou com dor no estômago e sensação de queimação constante há dois di...
              Sintomas: dor no estômago, cansaço

📋 INCON

## 10. Análise de Sintomas Extraídos

In [128]:
# Análise de frequência de sintomas
all_extracted = []
all_matched = []

for r in results:
    all_extracted.extend(r['sintomas_extraidos'])
    all_matched.extend([m['matched'] for m in r['match_details']])

print("\n" + "="*100)
print("ANÁLISE DE SINTOMAS EXTRAÍDOS")
print("="*100)

print(f"\nTotal de sintomas extraídos: {len(all_extracted)}")
print(f"Total de sintomas mapeados com sucesso: {len(all_matched)}")
print(f"Taxa de sucesso no mapeamento: {(len(all_matched)/len(all_extracted)*100):.1f}%")

print("\nSintomas mais frequentes (Top 15):")
symptom_freq = Counter(all_matched)
for symptom, count in symptom_freq.most_common(15):
    print(f"  • {symptom:30s} - {count:2d} ocorrências")


ANÁLISE DE SINTOMAS EXTRAÍDOS

Total de sintomas extraídos: 90
Total de sintomas mapeados com sucesso: 58
Taxa de sucesso no mapeamento: 64.4%

Sintomas mais frequentes (Top 15):
  • cansaço                        - 11 ocorrências
  • febre alta                     -  6 ocorrências
  • náuseas                        -  5 ocorrências
  • dificuldade para respirar      -  5 ocorrências
  • febre                          -  4 ocorrências
  • dor no peito                   -  4 ocorrências
  • ansiedade                      -  4 ocorrências
  • dor no tórax                   -  2 ocorrências
  • aperto no peito                -  2 ocorrências
  • dor de cabeça                  -  2 ocorrências
  • queimação                      -  2 ocorrências
  • dor no estômago                -  2 ocorrências
  • inchaço nos pés                -  2 ocorrências
  • dor pulsante na cabeça         -  1 ocorrências
  • desconforto abdominal          -  1 ocorrências


## 11. Exemplo Detalhado de Uma Análise Completa

In [129]:
# Mostrar análise completa de um exemplo
example_idx = 0  # Primeira frase
example = results[example_idx]

print("\n" + "="*100)
print("EXEMPLO DETALHADO DE ANÁLISE COMPLETA")
print("="*100)

print(f"\n📝 FRASE #{example['frase_id']}:")
print(f"{example['frase']}")

print(f"\n🔍 PROCESSAMENTO:")
print(f"   1. Sintomas extraídos (após lematização):")
for i, symp in enumerate(example['sintomas_extraidos'], 1):
    print(f"      {i}. {symp}")

print(f"\n   2. Mapeamento para base de conhecimento:")
for i, match in enumerate(example['match_details'], 1):
    print(f"      {i}. '{match['extracted']}' → '{match['matched']}' (Similaridade: {match['similarity_score']:.1%})")

print(f"\n   3. Diagnósticos candidatos (por score):")
for i, (diag, score) in enumerate(example['candidates'], 1):
    print(f"      {i}. {diag:30s} - Confiança: {score:5.1f}%")

print(f"\n✅ DIAGNÓSTICO SUGERIDO: {example['diagnostico_sugerido']}")
print(f"   Confiança: {example['confianca']:.1f}%")


EXEMPLO DETALHADO DE ANÁLISE COMPLETA

📝 FRASE #1:
Sinto uma dor forte no peito há algumas horas, com sensação de aperto e suor excessivo, o que me deixou sem conseguir realizar minhas atividades.

🔍 PROCESSAMENTO:
   1. Sintomas extraídos (após lematização):
      1. dor
      2. forte
      3. em o
      4. peito
      5. sensação

   2. Mapeamento para base de conhecimento:
      1. 'forte' → 'febre' (Similaridade: 60.0%)
      2. 'peito' → 'dor no peito' (Similaridade: 58.8%)
      3. 'sensação' → 'cansaço' (Similaridade: 66.7%)

   3. Diagnósticos candidatos (por score):
      1. Insuficiência Cardíaca         - Confiança: 100.0%
      2. Pneumonia                      - Confiança:  90.0%
      3. Infarto                        - Confiança:  88.2%

✅ DIAGNÓSTICO SUGERIDO: Insuficiência Cardíaca
   Confiança: 100.0%


## 12. Estatísticas Gerais

In [130]:
# Estatísticas finais
confidences = [r['confianca'] for r in results]
symptoms_counts = [r['num_sintomas'] for r in results]

print("\n" + "="*100)
print("ESTATÍSTICAS FINAIS DO SISTEMA")
print("="*100)

print(f"\n📊 CONFIANÇA DAS PREDIÇÕES:")
print(f"   Mínima:     {min(confidences):6.1f}%")
print(f"   Máxima:     {max(confidences):6.1f}%")
print(f"   Média:      {sum(confidences)/len(confidences):6.1f}%")
print(f"   Mediana:    {sorted(confidences)[len(confidences)//2]:6.1f}%")

print(f"\n🧬 SINTOMAS EXTRAÍDOS POR FRASE:")
print(f"   Mínimo:     {min(symptoms_counts):2d}")
print(f"   Máximo:     {max(symptoms_counts):2d}")
print(f"   Média:      {sum(symptoms_counts)/len(symptoms_counts):5.1f}")

print(f"\n📈 DIAGNÓSTICOS DISTRIBUIÇÃO:")
for diagnosis in sorted(diagnosis_groups.keys()):
    count = len(diagnosis_groups[diagnosis])
    percentage = (count / len(results)) * 100
    print(f"   {diagnosis:30s}: {count:2d} casos ({percentage:5.1f}%)")

print("\n" + "="*100)


ESTATÍSTICAS FINAIS DO SISTEMA

📊 CONFIANÇA DAS PREDIÇÕES:
   Mínima:        0.0%
   Máxima:      100.0%
   Média:        93.3%
   Mediana:     100.0%

🧬 SINTOMAS EXTRAÍDOS POR FRASE:
   Mínimo:      0
   Máximo:      5
   Média:        3.0

📈 DIAGNÓSTICOS DISTRIBUIÇÃO:
   Enxaqueca                     :  2 casos (  6.7%)
   Gastrite                      :  2 casos (  6.7%)
   Inconclusivo                  :  2 casos (  6.7%)
   Infarto                       :  4 casos ( 13.3%)
   Insuficiência Cardíaca        :  9 casos ( 30.0%)
   Pneumonia                     :  7 casos ( 23.3%)
   Transtorno de Ansiedade       :  4 casos ( 13.3%)



## 13. Exportar Resultados

In [131]:
# Exportar resultados para CSV usando configuração centralizada
output_df = pd.DataFrame([
    {
        'ID': r['frase_id'],
        'Frase': r['frase'],
        'Número_Sintomas_Extraídos': r['num_sintomas'],
        'Sintomas_Identificados': '; '.join([m['matched'] for m in r['match_details']]),
        'Diagnóstico_Sugerido': r['diagnostico_sugerido'],
        'Confiança_Percentual': round(r['confianca'], 2),
        'Diagnósticos_Alternativos': '; '.join([f"{d}({c:.0f}%)" for d, c in r['candidates'][1:3]]) if len(r['candidates']) > 1 else 'Nenhum'
    }
    for r in results
])

output_df.to_csv(RESULTS_CSV, index=False, encoding='utf-8')

print(f"✅ Resultados exportados para: {RESULTS_CSV}")
print(f"\nPrimeiras 5 linhas do arquivo de saída:")
print(output_df.head().to_string())

✅ Resultados exportados para: /Users/amanda/Documents/FIAP/projetos/chap01-phase02-automate-diagnostics/document/resultados_diagnostico.csv

Primeiras 5 linhas do arquivo de saída:
   ID                                                                                                                                              Frase  Número_Sintomas_Extraídos                      Sintomas_Identificados     Diagnóstico_Sugerido  Confiança_Percentual                   Diagnósticos_Alternativos
0   1  Sinto uma dor forte no peito há algumas horas, com sensação de aperto e suor excessivo, o que me deixou sem conseguir realizar minhas atividades.                          5                febre; dor no peito; cansaço   Insuficiência Cardíaca                 100.0                Pneumonia(90%); Infarto(88%)
1   2                             Estou com dor no tórax em forma de pressão desde hoje cedo, junto com falta de ar e cansaço extremo ao mínimo esforço.                          5          

## Resumo Executivo

### Técnicas NLP Aplicadas:

1. **Lematização com SPACY**: Reduz variações de palavras à forma canônica
   - Exemplo: "dor", "dores" → "dor"

2. **Remoção de Stopwords com NLTK**: Remove palavras com baixo significado semântico
   - Remove: "o", "a", "que", "de", etc.

3. **Named Entity Recognition (NER)**: Identifica entidades (embora em português é limitado)

4. **Análise Sintática com Regex e Pattern Matching**: Extrai estruturas de sintomas
   - Padrões: "dor no X", "sensação de Y", "falta de Z"

5. **Fuzzy Matching**: Encontra sintomas similares na base mesmo com pequenas variações
   - Usa SequenceMatcher para similaridade de strings

6. **Scoring e Ranking**: Sugere diagnóstico com base em evidências
   - Score normalizado entre 0-100%

### Fluxo do Sistema:
```
Frase → Extração de Sintomas → Lematização → Stopword Removal → 
 Fuzzy Matching → Busca no CSV → Scoring → Diagnóstico Sugerido
```

---

## 📚 Análises Avançadas: Ontologia e Grafo de Dependências

As técnicas avançadas de análise semântica foram movidas para um **notebook separado** para melhor organização e modularidade:

### 🔗 **[Ontologia_Grafo_Dependencias.ipynb](./Ontologia_Grafo_Dependencias.ipynb)**

Este notebook complementar contém:
- **Ontologia em RDF**: Estrutura formal das relações sintoma-diagnóstico (157 triplas)
- **Grafo de Dependências**: Modelagem com NetworkX (50 nós, 50 arestas)
- **Visualizações avançadas**: Grafo completo e subgrafos hierárquicos
- **Análise de centralidade**: Sintomas mais influentes
- **Estatísticas estruturais**: Densidade, conectividade, diâmetro

### ℹ️ Como usar:
1. Execute este notebook (`NLP_diagnostics.ipynb`) primeiro
2. Abra o notebook complementar `Ontologia_Grafo_Dependencias.ipynb`
3. O notebook complementar reutilizará os dados já carregados

Esta separação permite:
✓ Melhor organização de código  
✓ Menor tempo de execução do notebook principal  
✓ Análises opcionais (usuários podem pular se desejar)